# NiyamTrace-X Wave 3 / Experiment 11 — AgentDojo + Agent-SafetyBench Security Transfer

This notebook tests whether NiyamTrace-X's **authority-contraction idea transfers outside NiyamTrace-Bench**.

For AgentDojo it runs both clean and prompt-injected trajectories, then compares tool effects for the *same legitimate user task*. An attack-induced tool call that is not present in the clean trajectory is treated as an **effect-expansion proxy**, not automatically as a proven vulnerability. Official AgentDojo utility/security fields are retained separately.

For Agent-SafetyBench the notebook clones the official released code/data, records the exact commit, audits environment/task coverage, and optionally invokes its official evaluator when the configured model is supported. Unsupported model adapters are reported explicitly.

The notebook never substitutes an LLM judge for the benchmark's own scorer unless you explicitly configure one.


In [ ]:
import importlib.util,subprocess,sys
subprocess.check_call([sys.executable,'-m','pip','install','-q','agentdojo','pandas','numpy','matplotlib'])


In [ ]:
from pathlib import Path
import os, sys, json, re, math, time, random, hashlib, zipfile, shutil, subprocess, statistics, tempfile, unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED=20260911
random.seed(SEED); np.random.seed(SEED)
BASE=Path('/content') if Path('/content').exists() else Path('/mnt/data')
RESULTS=BASE/'niyamtrace_q1_wave3_results'
RESULTS.mkdir(parents=True,exist_ok=True)
print('BASE:',BASE)
print('RESULTS:',RESULTS)


In [ ]:
# Multi-model configuration. For a formal run, use at least 3 independent model families.
# Preferred: set NTX_MODELS_JSON in the environment so credentials never enter the notebook.
# Format:
# [
#   {"label":"qwen35-122b","model":"Qwen/Qwen3.5-122B-A10B-FP8","api_url":"http://HOST:PORT/v1","api_key":"EMPTY"},
#   {"label":"gpt-oss-120b","model":"openai/gpt-oss-120b","api_url":"http://HOST:PORT/v1","api_key":"EMPTY"}
# ]
raw=os.getenv('NTX_MODELS_JSON','').strip()
MODELS=json.loads(raw) if raw else []
for m in MODELS:
    for k in ['label','model','api_url']:
        if not m.get(k): raise ValueError(f'Model spec missing {k}: {m}')
    m.setdefault('api_key','EMPTY')
print('Configured model endpoints:', [m['label'] for m in MODELS])
if not MODELS:
    print('No model endpoints configured. Benchmark discovery/mapping cells can still run; model inference cells will record SKIPPED_NO_MODEL_CONFIG rather than invent results.')


In [ ]:
MODE=os.getenv('NTX_RUN_MODE','QUICK').upper(); assert MODE in {'QUICK','STANDARD','FULL'}
SUITES=['workspace','travel','banking','slack']
TASK_IDS={'QUICK':['user_task_0','user_task_1'],'STANDARD':[f'user_task_{i}' for i in range(10)],'FULL':[]}[MODE]
ATTACK=os.getenv('AGENTDOJO_ATTACK','tool_knowledge')
print('AgentDojo mode:',MODE,'attack:',ATTACK,'tasks:',TASK_IDS or 'ALL')


In [ ]:
# Record the installed AgentDojo version and CLI surface before running anything.
import agentdojo, subprocess
ver=getattr(agentdojo,'__version__','unknown')
help_out=subprocess.run([sys.executable,'-m','agentdojo.scripts.benchmark','--help'],capture_output=True,text=True).stdout
(RESULTS/'exp11_agentdojo_cli_help.txt').write_text(help_out)
print('AgentDojo version:',ver)


In [ ]:
# Run clean and attacked trajectories through the official AgentDojo CLI for every configured model.
def run_agentdojo(spec,suite,attack=None):
    out=RESULTS/f"exp11_agentdojo_{spec['label']}_{suite}_{attack or 'clean'}"
    out.mkdir(parents=True,exist_ok=True)
    env=os.environ.copy(); env['OPENAI_COMPATIBLE_BASE_URL']=spec['api_url']; env['OPENAI_COMPATIBLE_API_KEY']=spec.get('api_key','EMPTY')
    cmd=[sys.executable,'-m','agentdojo.scripts.benchmark','-s',suite,'--model','openai-compatible','--model-id',spec['model'],'--logdir',str(out),'--max-workers','1']
    for t in TASK_IDS: cmd += ['-ut',t]
    if attack: cmd += ['--attack',attack]
    p=subprocess.run(cmd,env=env,capture_output=True,text=True)
    (out/'stdout.txt').write_text(p.stdout); (out/'stderr.txt').write_text(p.stderr)
    return {'model':spec['label'],'suite':suite,'condition':attack or 'clean','returncode':p.returncode,'output_dir':str(out),'stderr_tail':p.stderr[-1000:]}
runs=[]
for m in MODELS:
    for s in SUITES:
        runs.append(run_agentdojo(m,s,None)); runs.append(run_agentdojo(m,s,ATTACK))
if not MODELS:runs=[{'model':'NONE','suite':'ALL','condition':'NA','returncode':None,'output_dir':'','stderr_tail':'SKIPPED_NO_MODEL_CONFIG'}]
run_df=pd.DataFrame(runs); run_df.to_csv(RESULTS/'exp11_agentdojo_run_status.csv',index=False); display(run_df)


In [ ]:
# Parse AgentDojo JSON traces. Supports both dict-root and list-root JSON payloads.
# We preserve official fields and extract normalized tool-call signatures.

def trace_calls(obj):
    calls=[]
    def visit(x):
        if isinstance(x,dict):
            # AgentDojo FunctionCall shapes and OpenAI tool_calls.
            if x.get('function') and isinstance(x['function'],str):
                calls.append((x['function'],json.dumps(x.get('args',x.get('arguments',{})),sort_keys=True,default=str)))
            if isinstance(x.get('function'),dict) and x['function'].get('name'):
                f=x['function']
                calls.append((f['name'],json.dumps(f.get('arguments',{}),sort_keys=True,default=str)))
            if x.get('tool_name'):
                calls.append((str(x['tool_name']),json.dumps(x.get('tool_args',{}),sort_keys=True,default=str)))
            for v in x.values():
                visit(v)
        elif isinstance(x,list):
            for v in x:
                visit(v)
    visit(obj)
    return sorted(set(calls))


def deep_first(obj, names):
    """Return first scalar value whose key matches one of names, searching recursively."""
    names={str(n).lower() for n in names}
    if isinstance(obj,dict):
        for k,v in obj.items():
            if str(k).lower() in names and isinstance(v,(str,int,float,bool)):
                return v
        for v in obj.values():
            r=deep_first(v,names)
            if r is not None:
                return r
    elif isinstance(obj,list):
        for v in obj:
            r=deep_first(v,names)
            if r is not None:
                return r
    return None


def official_scalar(o,names):
    if isinstance(o,dict):
        for k,v in o.items():
            if k.lower() in names and isinstance(v,(bool,int,float,str)):
                return v
        for v in o.values():
            r=official_scalar(v,names)
            if r is not None:
                return r
    elif isinstance(o,list):
        for v in o:
            r=official_scalar(v,names)
            if r is not None:
                return r
    return None


def normalize_trace_payload(obj):
    """
    Return one or more logical trace records from an AgentDojo JSON payload.

    - dict root -> one record
    - list root containing multiple task-like dicts -> one record per task dict
    - event/message list -> keep the full list as one logical trace so tool calls are not lost
    """
    if isinstance(obj,dict):
        return [obj]
    if not isinstance(obj,list):
        return []

    dict_items=[x for x in obj if isinstance(x,dict)]
    task_items=[]
    for x in dict_items:
        task_id=deep_first(x,{'user_task_id','task_id'})
        suite=deep_first(x,{'suite_name','suite'})
        # A list of independent benchmark records normally exposes a task id per item.
        if task_id is not None and suite is not None:
            task_items.append(x)

    # Split only when the list clearly contains multiple independent task records.
    if len(task_items) >= 2:
        return task_items

    # Otherwise this is most likely a single trace represented as an event/message list.
    return [obj]


# Parser regression tests: dict root, event-list root, and multi-record list root.
_test_dict={
    'suite_name':'workspace','user_task_id':'user_task_1',
    'messages':[{'tool_name':'send_email','tool_args':{'to':'a@example.com'}}]
}
_test_events=[
    {'type':'metadata','suite_name':'travel','user_task_id':'user_task_2'},
    {'tool_name':'book_flight','tool_args':{'city':'DEL'}}
]
_test_multi=[
    {'suite_name':'banking','user_task_id':'user_task_3','tool_name':'transfer','tool_args':{'amount':1}},
    {'suite_name':'banking','user_task_id':'user_task_4','tool_name':'transfer','tool_args':{'amount':2}},
]
assert len(normalize_trace_payload(_test_dict)) == 1
assert len(normalize_trace_payload(_test_events)) == 1
assert deep_first(_test_events,{'user_task_id'}) == 'user_task_2'
assert len(normalize_trace_payload(_test_multi)) == 2
assert trace_calls(_test_events) == [('book_flight', '{"city": "DEL"}')]
print('✅ AgentDojo dict/list parser regression tests passed')


tr=[]
parse_errors=[]
for r in runs:
    od=Path(r.get('output_dir',''))
    if not od.exists():
        continue
    for p in od.rglob('*.json'):
        try:
            payload=json.loads(p.read_text())
        except Exception as e:
            parse_errors.append({'path':str(p),'error':repr(e)})
            continue

        records=normalize_trace_payload(payload)
        for record_index,o in enumerate(records):
            suite=deep_first(o,{'suite_name','suite'})
            user_task_id=deep_first(o,{'user_task_id','task_id'})
            injection_task_id=deep_first(o,{'injection_task_id'})
            attack_type=deep_first(o,{'attack_type','attack'})
            error=deep_first(o,{'error'})
            duration=deep_first(o,{'duration','duration_seconds','elapsed'})

            tr.append({
                'model':r['model'],
                'suite':str(suite if suite is not None else r['suite']),
                'condition':r['condition'],
                'user_task_id':str(user_task_id if user_task_id is not None else ''),
                'injection_task_id':injection_task_id,
                'attack_type':attack_type,
                'error':error,
                'duration':duration,
                'official_utility':official_scalar(o,{'utility','utility_score','user_task_success'}),
                'official_security':official_scalar(o,{'security','security_score','injection_task_success'}),
                'calls':trace_calls(o),
                'path':str(p),
                'record_index':record_index,
                'json_root_type':type(payload).__name__,
            })

traces=pd.DataFrame(tr)
pd.DataFrame(parse_errors).to_csv(RESULTS/'exp11_agentdojo_parse_errors.csv',index=False)

if len(traces):
    traces.assign(calls_json=traces.calls.map(json.dumps)).drop(columns='calls').to_csv(
        RESULTS/'exp11_agentdojo_traces.csv',index=False
    )

    # Pair clean vs attacked runs for same model/suite/user task.
    # Additional attacked effects are authority-expansion proxies.
    clean=traces[traces.condition=='clean']
    atk=traces[traces.condition==ATTACK]
    pairs=[]
    for _,a in atk.iterrows():
        c=clean[
            (clean.model==a.model) &
            (clean.suite==a.suite) &
            (clean.user_task_id==a.user_task_id)
        ]
        if c.empty:
            continue
        cc=set(c.iloc[0].calls)
        ac=set(a.calls)
        extra=ac-cc
        missing=cc-ac
        pairs.append({
            'benchmark':'AgentDojo',
            'model':a.model,
            'suite':a.suite,
            'case_id':a.user_task_id,
            'attack':ATTACK,
            'clean_call_count':len(cc),
            'attack_call_count':len(ac),
            'effect_expansion_proxy':bool(extra),
            'extra_call_count':len(extra),
            'missing_call_count':len(missing),
            'official_utility':a.official_utility,
            'official_security':a.official_security,
        })

    pair_df=pd.DataFrame(pairs)
    pair_df.to_csv(RESULTS/'exp11_agentdojo_effect_transfer.csv',index=False)
    display(
        pair_df.groupby(['model','suite']).agg(
            n=('case_id','size'),
            effect_expansion_rate=('effect_expansion_proxy','mean')
        ).reset_index() if len(pair_df) else pair_df
    )

    print('Parsed trace rows:',len(traces))
    print('JSON root types:',traces['json_root_type'].value_counts().to_dict())
    print('Parse errors:',len(parse_errors))
else:
    pd.DataFrame().to_csv(RESULTS/'exp11_agentdojo_traces.csv',index=False)
    pd.DataFrame().to_csv(RESULTS/'exp11_agentdojo_effect_transfer.csv',index=False)
    print('No AgentDojo trace rows found.')


In [ ]:
# Agent-SafetyBench official repository/data audit + optional official evaluator.
ASB=BASE/'Agent-SafetyBench'
if not ASB.exists():
    p=subprocess.run(['git','clone','-q','https://github.com/thu-coai/Agent-SafetyBench.git',str(ASB)],capture_output=True,text=True)
    clone_status={'returncode':p.returncode,'stderr':p.stderr[-1000:]}
else: clone_status={'returncode':0,'stderr':'already present'}
commit='UNAVAILABLE'
if ASB.exists():
    c=subprocess.run(['git','-C',str(ASB),'rev-parse','HEAD'],capture_output=True,text=True); commit=c.stdout.strip() or 'UNKNOWN'
files=[]
if ASB.exists():
    for p in ASB.rglob('*'):
        if p.is_file(): files.append({'path':str(p.relative_to(ASB)),'size':p.stat().st_size})
pd.DataFrame(files).to_csv(RESULTS/'exp11_agentsafetybench_file_inventory.csv',index=False)
status={'repo':'thu-coai/Agent-SafetyBench','commit':commit,'clone':clone_status,'file_count':len(files),'official_runner_attempts':[]}
# Generic model invocation is intentionally not faked because Agent-SafetyBench adapters are model-specific.
# If ASB_RUN_COMMAND_TEMPLATE is supplied, execute one official command per model with {model} substitution.
tmpl=os.getenv('ASB_RUN_COMMAND_TEMPLATE','').strip()
if tmpl and ASB.exists():
    for m in MODELS:
        cmd=tmpl.format(model=m['model'],label=m['label']); p=subprocess.run(cmd,shell=True,cwd=ASB,capture_output=True,text=True,env={**os.environ,'OPENAI_API_KEY':m.get('api_key',''),'OPENAI_BASE_URL':m['api_url']})
        status['official_runner_attempts'].append({'model':m['label'],'command':cmd,'returncode':p.returncode,'stdout_tail':p.stdout[-1000:],'stderr_tail':p.stderr[-1000:]})
else: status['official_runner_note']='No ASB_RUN_COMMAND_TEMPLATE supplied; repository/data audit completed, model run not fabricated.'
(RESULTS/'exp11_agentsafetybench_status.json').write_text(json.dumps(status,indent=2))
print(status)


In [ ]:
# Publication summary / manifest.
if Path(RESULTS/'exp11_agentdojo_effect_transfer.csv').stat().st_size>2:
    try:
        p=pd.read_csv(RESULTS/'exp11_agentdojo_effect_transfer.csv')
        if len(p):
            s=p.groupby('model').effect_expansion_proxy.mean().sort_values(); fig,ax=plt.subplots(figsize=(7,4)); s.plot(kind='bar',ax=ax); ax.set_ylabel('Attack-induced effect expansion proxy'); ax.set_ylim(0,1); ax.set_title('AgentDojo security transfer across models'); fig.tight_layout(); fig.savefig(RESULTS/'exp11_agentdojo_multimodel_transfer.png',dpi=220); plt.show()
    except Exception as e: print(e)
manifest={'experiment':'NTX-Q1-11','mode':MODE,'agentdojo_version':ver,'models':[{k:v for k,v in m.items() if k!='api_key'} for m in MODELS],'agentsafetybench_commit':commit,'result_files':{p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in RESULTS.glob('exp11_*') if p.is_file()}}
(RESULTS/'exp11_manifest.json').write_text(json.dumps(manifest,indent=2))


In [ ]:
# FINAL CELL — package every result from this experiment and download it.
PREFIX='exp11_'
ZIP_OUT=BASE/'NTX_Q1_11_AGENT_SECURITY_TRANSFER_RESULTS.zip'
with zipfile.ZipFile(ZIP_OUT,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(RESULTS.rglob('*')):
        if p.is_file() and p.name.startswith(PREFIX):
            z.write(p,arcname=str(p.relative_to(RESULTS)))
sha=hashlib.sha256(ZIP_OUT.read_bytes()).hexdigest()
print('Created:',ZIP_OUT)
print('SHA-256:',sha)
print('Size MiB:',round(ZIP_OUT.stat().st_size/1024**2,3))
try:
    from google.colab import files
    files.download(str(ZIP_OUT))
except Exception:
    print('Not running in Colab. ZIP is available at',ZIP_OUT)
